# CSIS3754 - Question 2: Soccer Game Results Classification
## Main Mid-Year Examination 2025

## 2.1 - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the dataset
results = pd.read_csv('game_results.csv')
print('Dataset loaded successfully!')
results.head()

## 2.2 - Brief Summary of the Dataset

In [ ]:
# Number of records and features
print(f'Number of records:  {results.shape[0]}')
print(f'Number of features: {results.shape[1]}')

In [ ]:
# Data type for each feature
print('Data type for each feature:')
print(results.dtypes)

## 2.3 - Probability of 1st Goal Scored: First Half vs Second Half

In [ ]:
# Adjust column name to match your dataset (e.g. '1st Goal', 'First Goal Minute')
# We assume the column is named '1st Goal' and contains the minute of the first goal

goal_col = '1st Goal'   # <-- adjust if needed

# Drop rows where first goal is missing (no goal scored)
goals = results[results[goal_col].notna()].copy()
goals[goal_col] = pd.to_numeric(goals[goal_col], errors='coerce')
goals = goals[goals[goal_col].notna()]

total_goals = len(goals)

first_half  = goals[goals[goal_col] <= 45]
second_half = goals[goals[goal_col] >  45]

prob_first  = round(len(first_half)  / total_goals, 2)
prob_second = round(len(second_half) / total_goals, 2)

print(f'Total matches with a recorded first goal: {total_goals}')
print(f'First half  (0-45 min) goals:  {len(first_half)}  -> Probability: {prob_first}')
print(f'Second half (46-90 min) goals: {len(second_half)} -> Probability: {prob_second}')

In [ ]:
# Pie chart comparing probabilities
labels = ['First Half (0-45 min)', 'Second Half (46-90 min)']
sizes  = [prob_first, prob_second]
colors = ['steelblue', 'coral']

plt.figure(figsize=(6, 6))
plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct='%1.2f%%',
    startangle=140,
    wedgeprops=dict(edgecolor='white')
)
plt.title('Probability of 1st Goal: First Half vs Second Half', fontsize=12)
plt.tight_layout()
plt.show()

## 2.4 - Handle Missing Values

In [ ]:
# Check for missing values and show percentages
missing_count = results.isnull().sum()
missing_pct   = (missing_count / len(results)) * 100
missing_df    = pd.DataFrame({'Missing Count': missing_count, 'Missing %': missing_pct.round(2)})
missing_df    = missing_df[missing_df['Missing Count'] > 0]

print('Columns with missing values:')
print(missing_df)

In [ ]:
# MOTIVATION:
# - Numeric columns: fill with MEDIAN (robust to outliers in sports data)
# - Text/object columns: fill with MODE (most frequent category)
# - The '1st Goal' column may have NaN where no goal was scored;
#   we fill these with 0 to indicate no goal (or drop depending on context).

for col in results.columns:
    if results[col].isnull().sum() > 0:
        if results[col].dtype == 'object':
            mode_val = results[col].mode()[0]
            results[col].fillna(mode_val, inplace=True)
            print(f"'{col}' (text):    filled with mode  = '{mode_val}'")
        else:
            median_val = results[col].median()
            results[col].fillna(median_val, inplace=True)
            print(f"'{col}' (numeric): filled with median = {median_val}")

print('\nData after handling missing values:')
results.head()

In [ ]:
# Confirm all missing values are handled
print('Missing values remaining:')
print(results.isnull().sum())
print(f'\nTotal missing: {results.isnull().sum().sum()}')

## 2.5 - Remove the Date Feature

In [ ]:
# Remove Date column as it is not needed for the predictive model
date_cols = [col for col in results.columns if 'date' in col.lower() or 'Date' in col]
results.drop(columns=date_cols, inplace=True)

print(f'Removed date column(s): {date_cols}')
print(f'\nDataframe shape: {results.shape}')
results.head()

## 2.6 - Convert Text Values to Numeric

In [ ]:
# Step 1: Encode target label 'Man of the Match' using LabelEncoder
# Typical values: team name of winning side (home/away team name)
# OR 'Yes'/'No' - adjust accordingly

le_target = LabelEncoder()
results['Man of the Match'] = le_target.fit_transform(results['Man of the Match'].astype(str))
print(f"'Man of the Match' classes: {list(le_target.classes_)}")
print(f"Encoded as: {list(range(len(le_target.classes_)))}")
results.head()

In [ ]:
# Step 2: Categorical encode object columns with < 10 unique values
object_cols = results.select_dtypes(include='object').columns.tolist()

cat_cols = [col for col in object_cols if results[col].nunique() < 10]
print(f'Columns to label-encode (< 10 unique values): {cat_cols}')

le = LabelEncoder()
for col in cat_cols:
    results[col] = le.fit_transform(results[col].astype(str))
    print(f"  '{col}' encoded.")

results.head()

In [ ]:
# Step 3: One-hot encode object columns with >= 10 unique values
remaining_obj = results.select_dtypes(include='object').columns.tolist()
print(f'Columns to one-hot encode (>= 10 unique values): {remaining_obj}')

if remaining_obj:
    results = pd.get_dummies(results, columns=remaining_obj, drop_first=False)
    print('One-hot encoding applied.')
else:
    print('No columns needed one-hot encoding.')

print(f'\nShape after encoding: {results.shape}')
results.head()

In [ ]:
# Confirm all features are numeric
obj_remaining = results.select_dtypes(include='object').columns.tolist()
print(f'Object columns remaining: {obj_remaining if obj_remaining else "None - all features are numeric!"}')
print('\nData types:')
print(results.dtypes)

## 2.7 - Correlation Vector and Heatmap (excluding one-hot encoded features)

In [ ]:
# Identify one-hot encoded columns (bool or columns generated by get_dummies)
# Correlation vector: correlation of all non-OHE features with 'Man of the Match'

# Keep only original numeric columns (non-OHE)
bool_cols   = results.select_dtypes(include='bool').columns.tolist()
# Convert bool to int for numeric operations
results[bool_cols] = results[bool_cols].astype(int)

# Select non-OHE columns: exclude columns that were created by get_dummies
# (they typically follow the pattern 'original_col_value')
# A simple heuristic: columns present in original dataset before OHE
# For correlation vector we use all numeric non-OHE columns

ohe_pattern  = [col for col in results.columns if '_' in col and col not in results.columns[:20]]
non_ohe_cols = [col for col in results.columns if col not in ohe_pattern]

corr_vector = results[non_ohe_cols].corr()[['Man of the Match']].drop('Man of the Match')
corr_vector = corr_vector.sort_values('Man of the Match', ascending=False)

print('Correlation with "Man of the Match":')
print(corr_vector)

In [ ]:
# Heatmap of the correlation vector
plt.figure(figsize=(4, max(4, len(corr_vector) // 2)))
sns.heatmap(
    corr_vector,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    vmin=-1, vmax=1
)
plt.title('Correlation with Man of the Match', fontsize=12)
plt.tight_layout()
plt.show()

### 2.7.1 - Inference from Correlation Vector

In [ ]:
discussion_27 = """
INFERENCE FROM CORRELATION VECTOR AND HEATMAP:
===============================================

- Features with a higher positive correlation value (closer to +1) with
  'Man of the Match' are more likely to be associated with the home team
  winning the Man of the Match award.

- Features with a higher negative correlation (closer to -1) are more
  associated with the away team's player winning the award.

- Features closest to 0 have little predictive power for determining
  which team's player will receive Man of the Match.

- Highly correlated features (e.g. goals, possession, shots on target)
  suggest that match performance metrics are the strongest predictors.
  These features should contribute most to our classifier's decisions.
"""
print(discussion_27)

## 2.8 - Define X, y and Train/Test Split

In [ ]:
# Define X (features) and y (target)
X = results.drop(columns=['Man of the Match'])
y = results['Man of the Match']

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
# 80% training / 20% testing split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train dimensions: {X_train.shape}')
print(f'y_train dimensions: {y_train.shape}')
print(f'X_test dimensions:  {X_test.shape}')
print(f'y_test dimensions:  {y_test.shape}')

In [ ]:
# Feature scaling (StandardScaler)
# Important for Logistic Regression; also applied consistently to all models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('StandardScaler applied.')
print(f'X_train_scaled shape: {X_train_scaled.shape}')
print(f'X_test_scaled shape:  {X_test_scaled.shape}')

## 2.9 - Train Classifiers with K-Fold Cross-Validation (k=10)

In [ ]:
classifiers = {
    'Naive Bayes':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(),
    'Logistic Regression': LogisticRegression()
}

k = 10
cv_results = {}

print(f'K-Fold Cross-Validation Results (k={k}):')
print('='*60)

for name, clf in classifiers.items():
    acc_scores = cross_val_score(clf, X_train_scaled, y_train, cv=k, scoring='accuracy')
    f1_scores  = cross_val_score(clf, X_train_scaled, y_train, cv=k, scoring='f1_weighted')

    cv_results[name] = {
        'Mean Accuracy': acc_scores.mean(),
        'Std Accuracy':  acc_scores.std(),
        'Mean F1':       f1_scores.mean(),
        'Std F1':        f1_scores.std()
    }

    print(f'\n{name}:')
    print(f'  Training Accuracy: {acc_scores.mean():.4f} (+/- {acc_scores.std():.4f})')
    print(f'  F1 Score:          {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})')

print('\n' + '='*60)

In [ ]:
# Learning Curves for each classifier
def plot_learning_curve(estimator, title, X, y, cv=10):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', n_jobs=-1
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std  = np.std(train_scores,  axis=1)
    val_mean   = np.mean(val_scores,   axis=1)
    val_std    = np.std(val_scores,    axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, 'o--', color='blue',  label='Training Score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean,   'o-',  color='green', label='Cross-Validation Score')
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='green')
    plt.title(f'Learning Curve for the {title} Classifier', fontsize=12)
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy Score')
    plt.legend(loc='best')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

for name, clf in classifiers.items():
    plot_learning_curve(clf, name, X_train_scaled, y_train, cv=k)

## 2.10 - Best Model: Predictions and Evaluation

In [ ]:
# Select model with highest F1 score
best_name  = max(cv_results, key=lambda k: cv_results[k]['Mean F1'])
best_model = classifiers[best_name]

print(f'Best model (highest F1): {best_name}')
print(f"  Mean F1 Score: {cv_results[best_name]['Mean F1']:.4f}")

# Fit on full training set and predict
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

# Accuracy
test_acc = accuracy_score(y_test, y_pred)
print(f'\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_)
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

## 2.11 - Comparison with Dummy Classifier (Baseline)

In [ ]:
# Dummy classifier uses the most frequent class as its prediction (baseline)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_scaled, y_train)
y_dummy_pred = dummy.predict(X_test_scaled)

dummy_acc = accuracy_score(y_test, y_dummy_pred)
dummy_f1  = cross_val_score(dummy, X_train_scaled, y_train, cv=k, scoring='f1_weighted').mean()

print(f'Dummy Classifier (baseline) - most_frequent strategy:')
print(f'  Test Accuracy: {dummy_acc:.4f} ({dummy_acc*100:.2f}%)')
print(f'  Mean F1 Score: {dummy_f1:.4f}')
print(f'\nBest Model ({best_name}):')
print(f'  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f"  Mean F1 Score: {cv_results[best_name]['Mean F1']:.4f}")

In [ ]:
comparison_discussion = """
COMPARISON WITH DUMMY CLASSIFIER - DISCUSSION:
===============================================

The Dummy Classifier serves as a baseline by always predicting the most
frequent class, requiring no learning from the data.

- If the best model's accuracy and F1 score are significantly higher than
  the dummy classifier, this confirms that the model has learned meaningful
  patterns from the data and provides genuine predictive value.

- If the improvement is minimal (e.g. less than 5%), the model may not be
  adding much beyond simply guessing the dominant class. This could be due
  to class imbalance, insufficient features, or poor feature encoding.

Recommendation:
  A meaningful improvement over the dummy classifier validates that machine
  learning is worthwhile for this problem. If the improvement is small,
  consider feature engineering, addressing class imbalance (SMOTE), or
  trying ensemble methods such as Random Forest or Gradient Boosting.
"""
print(comparison_discussion)